# Practical 4A — IMDB CBOW Sentiment Analysis

Extracted from `4A.pdf`. Original displayed outputs omitted; executable Python code retained.

**Required local inputs:** `IMDB Dataset.csv`

In [1]:
# 1. Import necessary packages
import numpy as np
import pandas as pd
import re
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
# 2. Load the IMDB dataset
data = pd.read_csv('LP_IV_Updated Datasets/LP_IV_Updated Datasets/CBOW/IMDB Dataset.csv')
print(data.head())
# 3. Preprocess the text
stop_words = {'the', 'is', 'a', 'an', 'and', 'to', 'of', 'in', 'it', 'this', 'that'}
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return words
sentences = data['review'].apply(preprocess).tolist()
# Convert sentiment labels into numbers
labels = data['sentiment'].map({
    'positive': 1,
    'negative': 0
}).values
# 4. Split training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    sentences, labels,
    test_size=0.2,
    random_state=42
)
# 5. Train CBOW Word2Vec model
cbow_model = Word2Vec(
    sentences=X_train,
    vector_size=100,
    window=5,
    min_count=2,
    sg=0,
    epochs=10
)
# 6. Convert documents into embeddings
def document_vector(words):
    vectors = [
        cbow_model.wv[word]
        for word in words
        if word in cbow_model.wv
    ]
    if len(vectors) == 0:
        return np.zeros(100)
    return np.mean(vectors, axis=0)
X_train_vec = np.array([document_vector(x) for x in X_train])
X_test_vec = np.array([document_vector(x) for x in X_test])
# 7. Build Feedforward Neural Network
model = Sequential([
    Input(shape=(100,)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
# 8. Train the model
history = model.fit(
    X_train_vec, y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.1
)
# 9. Evaluate the model
loss, accuracy = model.evaluate(X_test_vec, y_test)
print("Test Loss:", loss)
print("Test Accuracy:", accuracy * 100, "%")


                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
Epoch 1/5
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - accuracy: 0.8341 - loss: 0.3751 - val_accuracy: 0.8595 - val_loss: 0.3272
Epoch 2/5
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.8541 - loss: 0.3387 - val_accuracy: 0.8608 - val_loss: 0.3233
Epoch 3/5
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.8577 - loss: 0.3308 - val_accuracy: 0.8515 - val_loss: 0.3382
Epoch 4/5
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - accuracy: 0.8592 - loss: 0.3278 - val_accuracy: 0.8652 - val_loss: 0.3177
Epoch 5/5
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.8612 - loss: 0.3233 - val_accuracy: 0